[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/epipolar/epipolar.ipynb)

# Two Views of a House

Two cameras photograph the same house from slightly different places. Each picture alone says only in which direction each corner lies, not how far away it is. Together they say much more: the two sight rays toward the same corner must cross in space, and that requirement alone fixes how the second camera is turned and in which direction it moved, and then where every corner is. Only the overall scale is lost: a house twice as large, seen from twice as far, gives the same pictures.

Each sight ray is a line, the join of the eye with an image point, and two lines cross exactly when their regressive product vanishes. That product, a number per pair of rays, is the residual to drive to zero over the motor that places the second camera. A twist moves a line by their commutator, so the residuals' rates under a small motion are linear forms on twists, and Gauss–Newton steps are solves against the sum of their squares, a form on two twists, measured by the second camera's kinetic energy. With the pose found, a ray's squared distance to a point is a quadric on points, and each corner is the vertex of its two rays' quadrics summed.

In the matrix notation of computer vision the crossing condition reads as the essential matrix constraint $\mathbf r_2^{\mathsf T} E\, \mathbf r_1 = 0$ with $E = [\mathbf t]_\times R$, usually solved by the eight-point algorithm and a singular value decomposition into four candidate poses, and the last step as linear triangulation.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Extensor, stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from itertools import islice

from examples.util.animation import show_animation
from examples.estimation.epipolar import render
from examples.estimation.epipolar.scenarios import HOUSE_EDGES, easing, house

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Line = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
Twist = ga.gatype.bivector()
Camera = ga.gatype((Point, Point))                                       # Point <- Point
# Camera 2 as a rigid body, by which a step is measured: unit masses a unit from its centre along each axis.
BODY = mv.point(np.concatenate([np.eye(3), -np.eye(3)]))                 # [masses] Point

## 1. Two pictures

Camera 1 sits at the origin and looks along z at the screen one unit away; a point's image is where the sight ray through it, its join with the eye, meets the screen. Camera 2 is the same camera, turned about y and shifted, so it sees the landmarks moved by the inverse of its pose. Both images get a little sensor noise, a displacement within the screen.

In [ ]:
# Camera 2's true pose, turned about y and then shifted; sensor noise on the screen at unit distance, about
# 1.5 pixels on a 1000 pixel sensor, and its seed.
turn = np.radians(14.0)
shift = np.array([0.65, 0.08, 0.18])
noise, seed = 0.0015, 42

In [ ]:
landmarks = house()                                                            # [landmarks] Point
origin = mv.w.dual()                                                           # [] Point
screen = mv.z - mv.w                                                           # [] Plane
camera = (origin & Point) ^ screen                                             # [] Point <- Point

true_shift = (stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0) / 2            # [] Twist
true_motor = (true_shift.exp() * (mv.zx * (turn / 2)).exp()).normalized()       # Motor

# Each camera images the landmarks in its own frame; dividing by the weight gives the image points unit weight.
projected = camera(Extensor.stack([landmarks, true_motor << landmarks]))       # [views, landmarks] Point
images = projected / (mv.w & projected)                                        # [views, landmarks] Point
in_screen = mv.direction([[1, 0, 0], [0, 1, 0]])                               # [2] Point
offsets = np.random.default_rng(seed).normal(0.0, noise, size=images.shape + (2,))
measured = images + (in_screen * offsets).sum(axis=-1)                         # [views, landmarks] Point
rays = (origin & measured).normalized()                                        # [views, landmarks] Line

In [ ]:
print("the camera, a map from points to their images:\n", camera.coefficient_table())

## 2. The pose from crossing rays

The search starts from the right baseline without any turn. Each step solves for the twist that best zeroes the regressive products of the paired rays; the length of the baseline is the one direction the pictures cannot see, and the solve drops it, so the baseline keeps the length it started with. With the pose found at any step, each landmark is triangulated as the point nearest both its sight rays.

In [ ]:
n_iterations = 4

In [ ]:
def refine(rays_1: Line, rays_2: Line, motor: Motor) -> Motor:
    """One Gauss-Newton step on the relative camera motor, in camera 1's frame, toward crossing sight rays.

    The rays are calibrated sight rays, each in its own camera's frame. The motor needs a nonzero
    baseline; the baseline's length is not observable and stays close to the given one.
    """
    # Camera 2's kinetic energy on twists in its own frame, to measure the step by.
    kinetic = Twist & (BODY & BODY.commutator(Twist)).sum(axis=0)        # [] Scalar <- (Twist, Twist)
    # Two lines meet (are coplanar) iff their regressive product vanishes. For two lines
    # it is a number, one residual per ray pair.
    residuals = rays_1 & (motor >> rays_2)                               # [rays] Scalar

    # A twist moves a line by their commutator, so the residual's rate of change under a
    # twist is a linear form on twists, its Jacobian.
    jacobian = rays_1 & Twist.commutator(motor >> rays_2)                # [rays] Scalar <- Twist

    # Gauss-Newton normal equations as forms on twists, summed over ray pairs. The residual
    # is already a number, so squaring it needs no metric: the curvature is the Jacobian
    # form times itself, and the gradient the residual times the Jacobian form.
    curvature = (jacobian * jacobian).sum(axis=0)                        # [] Scalar <- (Twist, Twist)
    descent = -(residuals * jacobian).sum(axis=0)                        # [] Scalar <- Twist

    # Solve for the 5 observable degrees of freedom, measuring a step by the kinetic energy
    # of camera 2. The flattest mode is the unobservable scale, which moves camera 2 along the
    # baseline; rcond drops it by its curvature per unit energy, and the step has no
    # component along it in that energy, so near the solution the baseline keeps its length.
    energy = kinetic(motor << Twist, motor << Twist)                     # [] Scalar <- (Twist, Twist)
    step = curvature.lstsq(descent, energy, rcond=1e-4)                  # [] Twist
    return ((step * 0.5).exp() * motor).normalized()                     # [] Motor


def triangulate(rays_1: Line, rays_2: Line, motor: Motor) -> Point:
    """The world points the sight rays of both cameras point at, in camera 1's frame, given camera 2's
    motor."""
    # ray & Point is the plane through a ray and an unknown point, and its squared norm is the point's
    # squared distance from the ray. The world point is the vertex of the distance quadrics summed over
    # both views. A dyad on the point's weight pins the scale without moving the vertex, so it is one solve.
    rays = Extensor.stack([rays_1, motor >> rays_2])                     # [views, rays] Line
    distances = (rays & Point).scalar_norm_squared().sum(axis=0)         # [rays] Scalar <- (Point, Point)
    weight = mv.w & Point                                                # [] Scalar <- Point
    return (distances + weight * weight).solve(weight).normalized()      # [rays] Point


def reconstruction(rays_1: Line, rays_2: Line, motor: Motor) -> Iterator[tuple[Motor, Point]]:
    """Jointly solve for the relative camera motor and the world points, from a first guess of the motor on,
    one Gauss-Newton step at a time, without end: each motor and the world points it implies."""
    while True:
        yield motor, triangulate(rays_1, rays_2, motor)
        motor = refine(rays_1, rays_2, motor)


iterates = islice(reconstruction(rays[0], rays[1], true_shift.exp().normalized()), n_iterations + 1)
est_motors, reconstructed = (Extensor.stack(path) for path in zip(*iterates))  # [iterations] Motor, [iterations, landmarks] Point
est_motor = est_motors[-1]

In [ ]:
def errors(landmarks: Point, reconstructed: Point, true_motor: Motor, est_motor: Motor):
    """How far the recovery is from the truth: the cosines between the true and found turns of the axis
    planes and between the baselines, and the house's RMS error once scaled about camera 1 to the true
    baseline, the baseline's length being unobservable."""
    axes = stack((mv.x, mv.y, mv.z))                                          # [3] Plane
    alignment = (est_motor >> axes).normalized() | (true_motor >> axes).normalized()   # [3] Scalar
    origin = mv.w.dual()
    true_baseline = origin & (true_motor >> origin)                           # [] Line
    est_baseline = origin & (est_motor >> origin)                             # [] Line
    agreement = true_baseline.normalized().dot(est_baseline.normalized())     # [] Scalar
    scale = true_baseline.norm() / est_baseline.norm()                        # [] Scalar
    unit = reconstructed / (mv.w & reconstructed)                             # [landmarks] Point
    # The difference of two unit points is a direction; its length is the norm of its complement.
    error = ((unit * scale + origin * (1.0 - scale)) - landmarks).dual()      # [landmarks] Plane
    return alignment, agreement, error.norm_squared().mean(axis=0).square_root()


alignment, agreement, rms = errors(landmarks, reconstructed[-1], true_motor, est_motor)

In [ ]:
print("turn error of each axis plane, degrees:", np.degrees(np.arccos(np.minimum(alignment.to_array(), 1))))
print("baseline direction error, degrees:", np.degrees(np.arccos(np.minimum(agreement.to_array(), 1))))
print("house error once scaled to the true baseline, RMS:", rms.to_array())

## 3. The search, animated

The animation eases camera 2 from one step of the search to the next along the screw between them, and at every pose triangulates the landmarks it implies. Camera 1 is blue; camera 2 is grey where it is and green where the search puts it. The true house is grey, its landmarks joined by its edges; the triangulated landmarks are coloured, with the sight rays of a few from both cameras, which cross at them. The pictures do not see the baseline's length, so what the search finds is drawn scaled about camera 1 to the true baseline, as the error above is measured. From the first guess, without any turn, the rays pass each other far apart and the triangulated points scatter; within three steps they settle onto the true ones, off only by the sensor noise.

In [ ]:
# The frames from one step to the next and held at the end, and a frame's duration in milliseconds.
n_easing, n_hold, frame_duration = 10, 16, 50

In [ ]:
show_animation(render.animate_reconstruction(landmarks, np.array(HOUSE_EDGES), true_motor, easing(rays, est_motors, n_easing, n_hold)),
               "epipolar_reconstruction", frame_duration)

In [ ]:
# checks
# The noise limits the recovery: each axis plane turns within half a degree of the truth, the baseline
# direction comes back within two degrees, and the house within 10 cm RMS.
assert np.all(alignment.to_array() > np.cos(np.radians(0.5)))
assert agreement.to_array() > np.cos(np.radians(2.0))
assert rms.to_array() < 0.1